# 02 - Exploratory analysis (EDA)

Works on `data/processed/transactions_clean.parquet` (output of `01_preprocessing`) with **DuckDB**, so the 10M rows are never loaded into memory all at once. Each analysis is a SQL query that returns a small pandas table, followed by a chart.

- `tx` = the full clean data (all 21 columns, read straight from the Parquet file)
- `base` = a narrow working table without the 4 free-text columns, stored in `data/interim/eda.duckdb` (built once, reused afterwards)
- Charts are saved to `reports/figures/` for the dashboard and slides.

**First time only:** `pip install duckdb matplotlib` inside `.venv`. Run the cells top to bottom.

## 0. Setup

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"Project root not found from {Path.cwd()}"

PARQUET = ROOT / "data" / "processed" / "transactions_clean.parquet"
DB_FILE = ROOT / "data" / "interim" / "eda.duckdb"  # working database (gitignored)
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert PARQUET.exists(), f"Missing {PARQUET} - run 01_preprocessing first"

con = duckdb.connect(str(DB_FILE))
con.execute("SET preserve_insertion_order = false")
# If the laptop runs out of memory, uncomment and lower this:
# con.execute("SET memory_limit = '3GB'")

con.execute(f"CREATE OR REPLACE VIEW tx AS SELECT * FROM read_parquet('{PARQUET.as_posix()}')")


def q(sql: str) -> pd.DataFrame:
    """Run a SQL query with DuckDB and return the result as a pandas DataFrame."""
    return con.sql(sql).df()


def save(fig, name: str) -> None:
    """Save a chart to reports/figures/<name>.png for the dashboard and slides."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("Project root:", ROOT)
print("DuckDB", duckdb.__version__)

### Working table `base`
All structured columns plus `ep` (timestamp as epoch seconds) and a `row_id`. The free-text columns (`*_details`, `*_tokens`) are left out to keep it fast; use the `tx` view when you need them. Building it takes a few minutes the first time only.

In [ ]:
tables = con.sql("SELECT table_name FROM information_schema.tables").df()["table_name"].tolist()
if "base" not in tables:
    con.execute("""
        CREATE TABLE base AS
        SELECT row_number() OVER ()           AS row_id,
               epoch_seconds                  AS ts,
               epoch(epoch_seconds)::BIGINT   AS ep,
               currency_code, usd_amount,
               ordering_bic8, beneficiary_bic8, bic8_path,
               total_duration, continuity_qflag,
               currency_conversion_count, participant_count, transaction_type_code,
               ordering_account, beneficiary_account,
               ordering_type, beneficiary_type,
               anomaly_type, is_anomaly
        FROM tx
    """)
n_base = q("SELECT count(*) AS n FROM base").loc[0, "n"]
print(f"{n_base:,} rows in base")

## 1. Sanity check
These numbers must match the profiling report. If an assert fails, the Parquet file is incomplete.

In [ ]:
checks = q("""
    SELECT count(*)            AS rows,
           count(anomaly_type) AS anomalies,
           sum(is_anomaly)     AS is_anomaly_sum,
           min(ts)             AS first_payment,
           max(ts)             AS last_payment,
           sum(usd_amount)     AS total_usd
    FROM base
""")
assert checks.loc[0, "rows"] == 10_000_000
assert checks.loc[0, "anomalies"] == 7_006 == checks.loc[0, "is_anomaly_sum"]
checks

## A. Anomaly types

Definitions from the case study: **ROMANCE_SCAM** (a consumer repeatedly sends money to a fake partner), **CEO_SCAM** (a business pays a fraudster after a spoofed CEO request), **ACCOUNT_REPLACEMENT_SCAM** (a real invoice with the account number swapped), **FAT_FINGER_MISTAKE** (comma separator misplaced in the amount), **DUPLICATE_PAYMENT** (accidental repeat). **CUSTOMER_RECALL** is in the data but *not defined* in the brief (question sent to the mentor).

### Hypotheses (written before testing)
| ID | Hypothesis | Test |
|---|---|---|
| H-A1 | Romance scams are sent only by consumers; CEO scams only by businesses | % consumer senders per type (A1) |
| H-A2 | A romance victim pays the same beneficiary several times in a short burst, with rising amounts | Payments per victim, time span, amount by payment number (A2) |
| H-A3 | A duplicate is an exact copy (same sender, receiver, amount, currency) sent shortly after the original, and repeats without the label are rare | Identical-payment groups, labelled vs not, time gap (A3) |
| H-A4 | A fat-finger amount is off from the sender's usual amount by a power of 10 | log10(amount / sender's normal median) vs normal payments (A4) |
| H-A5 | CEO and account-replacement scams go to a payee the sender has never paid normally | % of pairs with normal payment history (A5) |
| H-A6 | Scam rows are generated separately from normal traffic, so they lack the noise normal rows have | Path endpoint vs BIC mismatch rate (A6) |
| H-A7 | Scams take longer to settle, and this is not explained by the route | Duration by type, within the same transaction type (A7) |

### A1. Who sends and receives each anomaly type (H-A1)

In [ ]:
a1 = q("""
    SELECT coalesce(anomaly_type, '(normal)')                   AS anomaly_type,
           count(*)                                             AS n,
           round(100 * avg((ordering_type = 'C')::INT), 1)      AS pct_sender_consumer,
           round(100 * avg((beneficiary_type = 'C')::INT), 1)   AS pct_receiver_consumer
    FROM base
    GROUP BY 1
    ORDER BY n DESC
""")
a1

In [ ]:
pairs = q("""
    SELECT coalesce(anomaly_type, '(normal)') AS anomaly_type,
           ordering_type || '→' || beneficiary_type AS sender_to_receiver,
           count(*) AS n
    FROM base GROUP BY 1, 2
""").pivot(index="anomaly_type", columns="sender_to_receiver", values="n").fillna(0).astype(int)
pairs_pct = (pairs.div(pairs.sum(axis=1), axis=0) * 100).round(1)
pairs_pct  # % of each row (B = business, C = consumer)

In [ ]:
d = (a1.set_index("anomaly_type")[["pct_sender_consumer", "pct_receiver_consumer"]]
        .rename(columns={"pct_sender_consumer": "sender is a consumer",
                         "pct_receiver_consumer": "receiver is a consumer"}))
fig, ax = plt.subplots(figsize=(8, 4))
d.plot.barh(ax=ax, color=["#4C72B0", "#DD8452"])
ax.axvline(d.loc["(normal)", "sender is a consumer"], color="#4C72B0", ls="--", lw=1,
           label="normal payments: sender is a consumer")
ax.set_xlabel("% of payments")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.invert_yaxis()
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Romance scams are 100% sent by consumers, CEO scams 0%\n(dashed line = normal payments, 21%)")
save(fig, "A1_anomaly_party_types")
plt.show()

### A2. Romance scams: repeated payments to one beneficiary (H-A2)

In [ ]:
rom = q("""
    SELECT ordering_account, beneficiary_account, ep, usd_amount,
           row_number() OVER (PARTITION BY ordering_account, beneficiary_account ORDER BY ep, row_id) AS payment_no
    FROM base
    WHERE anomaly_type = 'ROMANCE_SCAM'
""")
per_victim = (rom.sort_values("payment_no")
                 .groupby(["ordering_account", "beneficiary_account"])
                 .agg(n_payments=("usd_amount", "size"),
                      first_usd=("usd_amount", "first"),
                      last_usd=("usd_amount", "last"),
                      span_min=("ep", lambda s: (s.max() - s.min()) / 60),
                      strictly_rising=("usd_amount", lambda s: bool((s.diff().dropna() > 0).all())))
                 .reset_index())
summary = pd.Series({
    "romance payments": len(rom),
    "victim -> beneficiary pairs": len(per_victim),
    "distinct victims (senders)": rom["ordering_account"].nunique(),
    "distinct beneficiaries": rom["beneficiary_account"].nunique(),
    "payments per victim (min / median / max)": f"{per_victim.n_payments.min()} / {per_victim.n_payments.median():.0f} / {per_victim.n_payments.max()}",
    "time span per victim, minutes (median)": round(per_victim.span_min.median(), 1),
    "time span per victim, minutes (90th pct)": round(per_victim.span_min.quantile(0.9), 1),
    "% of consecutive payments larger than the previous one": round(100 * (rom.sort_values("payment_no").groupby(["ordering_account", "beneficiary_account"]).usd_amount.diff().dropna() > 0).mean(), 1),
    "% of victims whose amounts rise every single time": round(100 * per_victim.strictly_rising.mean(), 1),
    "last / first amount (median)": round((per_victim.last_usd / per_victim.first_usd).median(), 1),
})
summary.to_frame("value")

In [ ]:
victims_per_bene = rom.groupby("beneficiary_account")["ordering_account"].nunique()
victims_per_bene.value_counts().sort_index().rename_axis("victims per beneficiary").to_frame("beneficiaries")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
per_victim.n_payments.value_counts().sort_index().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_xlabel("payments per victim")
axes[0].set_ylabel("victims")
axes[0].set_title("Each victim pays 2-8 times")

by_no = rom.groupby("payment_no")["usd_amount"].quantile([0.25, 0.5, 0.75]).unstack()
axes[1].plot(by_no.index, by_no[0.5], marker="o", color="#C44E52", label="median")
axes[1].fill_between(by_no.index, by_no[0.25], by_no[0.75], alpha=0.2, color="#C44E52", label="25th-75th pct")
axes[1].set_yscale("log")
axes[1].set_xlabel("payment number within the victim's sequence")
axes[1].set_ylabel("USD amount (log scale)")
axes[1].set_title("The median amount rises with each payment number")
axes[1].legend()
fig.suptitle("Romance scams: short bursts of escalating payments to one beneficiary", y=1.02)
save(fig, "A2_romance_bursts")
plt.show()

### A3. Duplicate payments: identical copies (H-A3)

In [ ]:
dup = q("""
    WITH g AS (
        SELECT ordering_account, beneficiary_account, usd_amount, currency_code,
               count(*)                                                AS n_copies,
               count(*) FILTER (WHERE anomaly_type = 'DUPLICATE_PAYMENT') AS n_dup_label,
               max(ep) - min(ep)                                       AS spread_s
        FROM base
        GROUP BY ALL
        HAVING count(*) > 1
    )
    SELECT *,
           CASE WHEN n_dup_label = n_copies THEN 'all copies labelled DUPLICATE'
                WHEN n_dup_label = 0        THEN 'no copy labelled DUPLICATE'
                ELSE 'partly labelled' END AS group_kind
    FROM g
""")
(dup.groupby("group_kind")
    .agg(groups=("n_copies", "size"), rows=("n_copies", "sum"),
         copies_min=("n_copies", "min"), copies_max=("n_copies", "max"),
         spread_s_median=("spread_s", "median"), spread_s_max=("spread_s", "max")))

In [ ]:
labelled = dup[dup.group_kind == "all copies labelled DUPLICATE"]
print("Copies per duplicate group:")
print(labelled.n_copies.value_counts().sort_index().to_string())
n_dup_total = q("SELECT count(*) AS n FROM base WHERE anomaly_type = 'DUPLICATE_PAYMENT'").loc[0, "n"]
print(f"\nDUPLICATE_PAYMENT rows inside these groups: {labelled.n_dup_label.sum()} of {n_dup_total}")

In [ ]:
# How well does "identical copy within 2 hours" separate labelled duplicates from unlabelled repeats?
window = 2 * 3600
(dup.assign(within_2h=dup.spread_s <= window)
    .groupby("group_kind")
    .agg(groups=("within_2h", "size"), within_2h=("within_2h", "sum"),
         spread_s_min=("spread_s", "min"), mean_usd=("usd_amount", "mean")))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, np.log10(dup.spread_s.max() + 1), 40)
for kind, color in [("all copies labelled DUPLICATE", "#C44E52"), ("no copy labelled DUPLICATE", "#4C72B0")]:
    ax.hist(dup.loc[dup.group_kind == kind, "spread_s"] + 1, bins=bins, alpha=0.6, label=kind, color=color)
ax.set_xscale("log")
ax.set_xlabel("time between first and last identical copy (seconds, log scale)")
ax.set_ylabel("groups")
ax.legend()
ax.axvline(2 * 3600, color="black", ls="--", lw=1)
ax.set_title("Labelled duplicates all repeat within 2 hours (dashed line);\nmost unlabelled repeats are weeks apart")
save(fig, "A3_duplicate_time_gap")
plt.show()

### A4. Fat-finger mistakes: amount off by a power of 10 (H-A4)

In [ ]:
ff = q("""
    WITH med AS (
        SELECT ordering_account, median(usd_amount) AS sender_median_usd
        FROM base
        WHERE anomaly_type IS NULL
        GROUP BY 1
    )
    SELECT coalesce(b.anomaly_type, '(normal sample)') AS kind,
           b.usd_amount, m.sender_median_usd,
           log10(b.usd_amount / m.sender_median_usd)   AS log10_ratio
    FROM base b
    JOIN med m USING (ordering_account)
    WHERE b.anomaly_type = 'FAT_FINGER_MISTAKE'
       OR (b.anomaly_type IS NULL AND b.row_id % 200 = 0)   -- 0.5% sample of normal payments as baseline
""")
ff["magnitude"] = ff.log10_ratio.round().astype(int)
pd.crosstab(ff.magnitude, ff.kind, normalize="columns").mul(100).round(1)  # % of each column

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.arange(-6, 9.5, 0.25)
for kind, color in [("(normal sample)", "#4C72B0"), ("FAT_FINGER_MISTAKE", "#C44E52")]:
    ax.hist(ff.loc[ff.kind == kind, "log10_ratio"], bins=bins, density=True, alpha=0.6, label=kind, color=color)
ax.set_xlabel("log10( amount / sender's usual amount )   (1 = 10x larger, -1 = 10x smaller)")
ax.set_ylabel("density")
ax.legend()
ax.set_title("Fat-finger amounts are mostly 100x-10,000x off the sender's usual amount;\nnormal payments stay within about 10x")
save(fig, "A4_fat_finger_magnitude")
plt.show()

### A5. New payee: has the sender ever paid this beneficiary normally? (H-A5)

In [ ]:
a5 = q("""
    WITH s AS (
        SELECT anomaly_type,
               count(*) FILTER (WHERE anomaly_type IS NULL)
                   OVER (PARTITION BY ordering_account, beneficiary_account)
               - (anomaly_type IS NULL)::INT AS other_normal_payments_same_pair
        FROM base
    )
    SELECT coalesce(anomaly_type, '(normal)') AS anomaly_type,
           count(*) AS n,
           round(100 * avg((other_normal_payments_same_pair > 0)::INT), 1) AS pct_pair_has_normal_history
    FROM s
    GROUP BY 1
    ORDER BY pct_pair_has_normal_history
""")
a5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
a5.set_index("anomaly_type")["pct_pair_has_normal_history"].plot.barh(ax=ax, color="#55A868")
ax.set_xlabel("% of payments where the same sender also paid the same receiver normally")
ax.set_ylabel("")
ax.set_title("CEO, account-replacement and romance scams go to payees never paid normally")
save(fig, "A5_new_payee")
plt.show()

### A6. Path endpoints vs BIC columns (H-A6)
In normal rows the first bank in `bic8_path` sometimes differs from `ordering_bic8`, and the last from `beneficiary_bic8`. Does the same happen in anomalies?

In [ ]:
a6 = q("""
    SELECT coalesce(anomaly_type, '(normal)') AS anomaly_type,
           count(*) AS n,
           round(100 * avg((split_part(bic8_path, ';', 1) <> ordering_bic8)::INT), 2)          AS pct_first_hop_differs,
           round(100 * avg((string_split(bic8_path, ';')[-1] <> beneficiary_bic8)::INT), 2)    AS pct_last_hop_differs
    FROM base
    GROUP BY 1
    ORDER BY n DESC
""")
a6

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(a6.set_index("anomaly_type")[["pct_first_hop_differs", "pct_last_hop_differs"]]
   .rename(columns={"pct_first_hop_differs": "first bank in path ≠ ordering BIC",
                    "pct_last_hop_differs": "last bank in path ≠ beneficiary BIC"})
   .plot.barh(ax=ax, color=["#8172B3", "#937860"]))
ax.set_xlabel("% of payments")
ax.set_ylabel("")
ax.invert_yaxis()
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Romance, CEO and account-replacement scams never have a last-hop mismatch;\nnormal rows do 5% of the time")
save(fig, "A6_path_mismatch")
plt.show()

### A7. Duration by anomaly type, controlling for the route (H-A7)

In [ ]:
a7 = q("""
    SELECT coalesce(anomaly_type, '(normal)') AS anomaly_type,
           transaction_type_code,
           count(*) AS n,
           median(total_duration) AS median_duration_s
    FROM base
    GROUP BY 1, 2
""")
a7["cell"] = a7.median_duration_s.round().astype(int).astype(str) + "  (n=" + a7.n.astype(str) + ")"
# median seconds (number of payments); compare each row with (normal) in the same column
a7.pivot(index="anomaly_type", columns="transaction_type_code", values="cell")

In [ ]:
direct = q("""
    SELECT coalesce(anomaly_type, '(normal)') AS anomaly_type, total_duration
    FROM base
    WHERE transaction_type_code = 'Direct'
      AND (anomaly_type IS NOT NULL OR row_id % 200 = 0)   -- all anomalies + 0.5% sample of normal
""")
order = direct.groupby("anomaly_type").total_duration.median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([direct.loc[direct.anomaly_type == k, "total_duration"] for k in order],
           vert=False, showfliers=False)
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels(order)
ax.set_xscale("log")
ax.set_xlabel("total duration, seconds (log scale) - Direct payments only")
ax.set_title("On Direct routes, the median scam payment takes 12-28x longer than a normal one (39 s)")
save(fig, "A7_duration_by_type")
plt.show()

### A8. Results → draft rules (verified on all 10,000,000 rows)

| ID | Rule (condition → effect → size → rows covered) | Evidence | Confidence |
|---|---|---|---|
| R-A1 | ROMANCE_SCAM → sender is a **Consumer in 100%** of rows (2,476/2,476; normal 21.2%). CEO_SCAM → sender is a **Business in 100%** (945/945). The other 4 types copy the normal mix (19.7-22.4% consumer senders). | A1 | High |
| R-A2 | ROMANCE_SCAM = **500 victims, each paying exactly 1 beneficiary 2-8 times** (median 5), within **18 min** (median; 90% within 40 min). **83.1%** of consecutive payments are larger than the previous one; the median amount rises from **$33 (1st) to $1,925 (8th)**. 378 beneficiaries, 90 of them hit by 2-6 victims. | A2 | High |
| R-A3 | DUPLICATE_PAYMENT = **185 groups of 2-5 identical copies** (same sender account, receiver account, USD amount, currency), **all copies labelled, including the first** (679/679), all within **203 s-2 h** (median 67 min). | A3 | High |
| R-A4 | FAT_FINGER_MISTAKE amounts are off from the sender's usual (median normal) amount by **≥ ~30x in 71.5%** of rows (normal: 11.5%); **72.5% larger, 20.5% smaller**. Consistent with a misplaced separator shifting the value by powers of 10. The 12 largest payments in the dataset are all fat-finger. | A4 | Medium: the intended amount is not in the data, so the exact shift can't be measured per row |
| R-A5 | CEO, account-replacement and romance scams go to a payee the sender **never paid normally** (0.2% / 0% / 0% have pair history, vs 23.6% of normal payments). Fat-finger, duplicate and recall rows (19.6-26.8%) look like normal relationships. | A5 | High |
| R-A6 | Romance, CEO and account-replacement rows **never** have a last-hop mismatch (0%, vs 4.99% of normal rows); romance rows never have a first-hop mismatch either. *Interpretation:* the fraudster side of these rows is generated separately, without the endpoint noise added to normal traffic. | A6 | High (observation) / Medium (interpretation) |
| R-A7 | On the same route type, scams settle much slower: Direct median **39 s normal vs 454 s (account replacement), 648 s (CEO), 1,102 s (romance)**; same pattern on Indirect. Fat-finger, recall and duplicate ≈ normal (40-50 s). | A7 | High for Direct/Indirect; Covered has only 3-7 scam rows |

**Hypotheses that did not hold (report these - the rubric rewards them):**
- H-A2 "amounts rise every time": only **44.8%** of victims have strictly rising amounts; the rise is a trend, not a rule.
- H-A1 / brief: romance beneficiaries are "another Consumer" per the brief, but **26.9% go to Businesses**.
- H-A3 "unlabelled repeats are rare": rejected. **2,439 unlabelled identical-repeat groups** exist (median 21 days apart); **139 of them repeat within 2 h**, so the time gap alone does not fully reproduce the label. Open question: what else separates them?
- CUSTOMER_RECALL: not defined in the brief; no difference from normal found in party type, payee history, path or duration so far.